# Riwaq | رواق
Bilingual fictional campus assistant · Hanan Ahmed Alahmadi

Choose **Runtime → Run all**. Default: no-secret, deterministic simulator through the real application and SDK mock transport. This demonstrates behavior, not model quality. Internet is needed for checkout/dependencies. No GPU is required.
The live conversation appears below the safety checks. Optional Hugging Face model evaluation is disabled by default.


In [1]:
import json, os, sys, subprocess
from pathlib import Path
# Use an existing checkout, or clone the real repository in Colab.
ROOT = Path(os.environ.get('RIWAQ_ROOT', '/content/riwaq' if Path('/content').exists() else '.')).resolve()
if not (ROOT / 'src/riwaq.py').exists():
    subprocess.run(['git', 'clone', 'https://github.com/hanan27/riwaq.git', str(ROOT)], check=True)
assert (ROOT / 'configs/models.json').is_file()
if os.environ.get('RIWAQ_SKIP_INSTALL') != '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements-demo.txt')], check=True)
sys.path.insert(0, str(ROOT / 'src'))
import riwaq
assert Path(riwaq.__file__).resolve() == ROOT / 'src/riwaq.py'
from backends import configuration
config = configuration()
print('Executed checkout:', ROOT)
subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], check=True)
print(json.dumps(config, indent=2))


Executed checkout: /content/riwaq
{
  "offline": {
    "url": "https://offline.invalid/v1",
    "model": "riwaq-rule-simulator"
  },
  "open_weight": {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "revision": "main",
    "context_limit": 4096
  },
  "hardware_hourly_usd": 1.0,
  "hosted": {
    "model": "Qwen/Qwen3-235B-A22B-Instruct-2507:deepinfra",
    "url": "https://router.huggingface.co/v1",
    "fallback_model": "Qwen/Qwen3-235B-A22B-Instruct-2507:novita",
    "prices_usd_per_million": [
      0.09,
      null,
      0.55
    ],
    "fallback_prices_usd_per_million": [
      0.09,
      null,
      0.58
    ],
    "catalog_source": "https://router.huggingface.co/v1/models",
    "catalog_checked_at": "2026-09-15T10:39:46.485440+00:00",
    "capabilities": {
      "deepinfra": {
        "status": "live",
        "supports_tools": true,
        "supports_structured_output": true,
        "pricing": {
          "input": 0.09,
          "output": 0.55
        }
      },
      "novit

## Deterministic safety checks
Simulated responses test production code, including the unchanged R079 golden expectation.

In [2]:
import unittest
suite = unittest.defaultTestLoader.discover(str(ROOT / 'tests'))
checks = unittest.TextTestRunner(verbosity=1).run(suite)
assert checks.wasSuccessful()
from evidence import case_session
from riwaq import CampusApp, Session, configured_client, language
case = next(c for c in json.loads((ROOT / 'data/golden.v1.json').read_text()) if c['id'] == 'R079')
app = CampusApp()
result = app.respond(case['text'], case_session(case))
assert result['status'] == case['expected'] == 'refused' and not app.tools.bookings
print('R079:', result)


................

PASS anonymous booking denied
PASS missing consent denied
PASS wrong role denied
PASS identity argument injection denied
PASS unknown tool denied
PASS loop overflow denied
PASS different slot consent denied
PASS idempotent replay and cross-student collision
PASS poisoned model/tool-content output 1
PASS poisoned model/tool-content output 2
PASS poisoned model/tool-content output 3
PASS poisoned model/tool-content output 4
PASS poisoned model/tool-content output 5
PASS indirect injection from tool 1
PASS indirect injection from tool 2
PASS indirect injection from tool 3
PASS indirect injection from tool 4
PASS indirect injection from tool 5


......................
----------------------------------------------------------------------
Ran 38 tests in 9.106s

OK


R079: {'status': 'refused', 'answer': 'لا أستطيع المساعدة في هذا الطلب. يرجى استخدام قناة دعم الطلاب الرسمية.', 'extraction': [{'stage': 'extract.v1', 'valid': True}]}


## Live bilingual conversation — simulator
These responses are generated now by the deterministic simulator using `CampusApp.respond`.
The fictional student session below grants consent only for Monday 9. It is test identity, not real authentication.
After Run all, enter Arabic or English text and press Send. Booking requires the explicit consent checkbox.


In [3]:
app = CampusApp(cache=True)
for text, session in [('What are the admissions documents?', Session()),
                      ('كم رسوم السجل الأكاديمي؟', Session()),
                      ('Book Monday 9', Session('student-demo', ('student',), 'mon-09')),
                      ('أحتاج موظف', Session())]:
    print('You:', text)
    print('Riwaq:', app.respond(text, session))
import ipywidgets as widgets
from IPython.display import display
message = widgets.Text(placeholder='Ask in Arabic or English', description='Message:')
slot = widgets.Dropdown(options=['mon-09', 'tue-11'], description='Slot:')
consent = widgets.Checkbox(value=False, description='I confirm this booking slot')
send = widgets.Button(description='Send')
conversation = widgets.Output()
def reply(_):
    session = Session('student-demo', ('student',), slot.value if consent.value else None)
    with conversation:
        print('You:', message.value)
        print('Riwaq:', app.respond(message.value, session)['answer'])
    consent.value = False
send.on_click(reply)
display(widgets.VBox([message, slot, consent, send, conversation]))


You: What are the admissions documents?
Riwaq: {'status': 'answered', 'answer': 'Admissions require a school certificate and an identity document.', 'source_id': 'NAM-AD-1'}
You: كم رسوم السجل الأكاديمي؟
Riwaq: {'status': 'answered', 'answer': 'رسوم السجل الأكاديمي الرسمي 25 ريال ومدة إصداره يومان عمل.', 'source_id': 'NAM-TR-1'}
You: Book Monday 9
Riwaq: {'status': 'booked', 'answer': 'Booked: mon-09', 'booking_id': 'B-d071f9f1', 'slot': 'mon-09', 'extraction': [{'stage': 'extract.v1', 'valid': True}]}
You: أحتاج موظف
Riwaq: {'status': 'handoff', 'answer': 'تم تحويل الطلب للدعم.', 'handoff_id': 'H-1', 'terminal': True}


## Production request stages
The following cell calls the same five functions used by `respond`. Production prompt text lives only in `prompts/`; metering records prompt versions and hashes.

In [4]:
stage_app = CampusApp()
text = 'What are the admissions documents?'
safe, layer = stage_app.stage_input(text)
assert safe
intent = stage_app.stage_route(text)
source = stage_app.stage_context(text)
raw = stage_app.stage_execute(text, intent, source, Session())
final = stage_app.stage_output(raw, language(text), source, intent)
print({'input': layer, 'route': intent, 'context': source, 'execute': raw, 'output': final})
print('Prompt evidence:', [{k: r.get(k) for k in ('prompt', 'prompt_sha256')} for r in stage_app.client.logs])


{'input': 'allow', 'route': 'faq', 'context': {'id': 'NAM-AD-1', 'en': 'Admissions require a school certificate and an identity document.', 'ar': 'يتطلب القبول شهادة الثانوية ووثيقة الهوية.'}, 'execute': {'status': 'answered', 'answer': 'Admissions require a school certificate and an identity document.', 'source_id': 'NAM-AD-1'}, 'output': {'status': 'answered', 'answer': 'Admissions require a school certificate and an identity document.', 'source_id': 'NAM-AD-1'}}
Prompt evidence: [{'prompt': 'faq.v1', 'prompt_sha256': 'e2ea0232feaa6f775d8066e10bceba42891aa1964fa6952e4a318af44564abe3'}]


## Optional real Hugging Face evaluation
Set one or both flags below and rerun this cell. Local weights need a model download and preferably a T4 GPU; no token is needed for public weights. Hosted inference needs a private `HF_TOKEN` Colab Secret with notebook access and may incur Hugging Face charges. The existing OpenAI SDK is only the HF-compatible transport; no OpenAI service is used.
Missing hosted credentials produce NOT RUN, without benchmark error rows. Review labels independently before adding reviewer/approval/date to `data/judge-calibration.v1.json`. Unreviewed reference agreement is not human calibration. Failed predictions have no kappa.


In [5]:
RUN_HOSTED = False
RUN_LOCAL_MODEL = False
if RUN_HOSTED or RUN_LOCAL_MODEL:
    if not os.getenv('HF_TOKEN'):
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
            if token: os.environ['HF_TOKEN'] = token
        except Exception:
            pass
    if RUN_LOCAL_MODEL:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)
    from evaluate import run
    aliases = tuple(name for name, enabled in [('hosted', RUN_HOSTED), ('open_weight', RUN_LOCAL_MODEL)] if enabled)
    result = run(ROOT, aliases=aliases)
    from IPython.display import Markdown
    display(Markdown((ROOT / 'EVALUATION_REPORT.md').read_text()))
    print('Reports saved in', ROOT, '; pending:', result['pending'])
else:
    print('Real model comparison: NOT RUN. Simulator output is not model evaluation.')


Real model comparison: NOT RUN. Simulator output is not model evaluation.


## Before resubmission
Inspect every executed cell. If running real evaluation, download `EVALUATION_REPORT.md` and `run-results.json` from the Colab Files panel under `riwaq`, and inspect failed gates and pending evidence. Save/download the executed notebook. Never paste credentials into cells or commit secrets. Default Run all does not overwrite model reports.
